# ex04 · 权重衰减与 Dropout（对应教材 4.5 权重衰减 / 4.6 暂退法）

> **做题流程**：补全 TODO，每个自测跑出 ✓；做完再看 `solutions/ex04-答案.md`。
>
> 难度标记：🌱 基础（预测+验证）｜🔧 变式（改动观察）｜🚀 挑战（闭卷复现）
>
> 本节讲两种正则化手段：权重衰减（L2）和 Dropout，都是对抗过拟合的武器（面试高频）。

In [ ]:
import torch
from torch import nn
from torch.utils import data

torch.manual_seed(0)

# 高维线性回归：200 个特征、只有 20 个训练样本 → 天然过拟合
n_train, n_test, num_inputs, batch_size = 20, 100, 200, 5
true_w = torch.ones((num_inputs, 1)) * 0.01
true_b = 0.05

def synthetic_data(w, b, n):
    X = torch.normal(0, 1, (n, len(w)))
    y = torch.matmul(X, w) + b
    y += torch.normal(0, 0.01, y.shape)
    return X, y.reshape((-1, 1))

train_features, train_labels = synthetic_data(true_w, true_b, n_train)
test_features, test_labels = synthetic_data(true_w, true_b, n_test)
train_iter = data.DataLoader(data.TensorDataset(train_features, train_labels), batch_size, shuffle=True)
test_iter = data.DataLoader(data.TensorDataset(test_features, test_labels), batch_size, shuffle=False)
print('训练样本:', n_train, ' 特征数:', num_inputs)

## 第一部分 · 权重衰减（L2 正则化）

### 题 1 🔧 从零实现 L2 惩罚项（TODO 4.1）

权重衰减就是在损失后面加一个惩罚项：L + (λ/2)·‖w‖²。先补全 l2_penalty。

In [ ]:
def l2_penalty(w):
    # TODO 4.1: torch.sum(w.pow(2)) / 2
    raise NotImplementedError('⚠ TODO 4.1: l2_penalty 未完成')

In [ ]:
try:
    assert abs(l2_penalty(torch.tensor([3.0, 4.0])) - 12.5) < 1e-6
    print('✓ l2_penalty 正确：(3²+4²)/2 = 12.5')
except NotImplementedError as e:
    print(f'⚠ {e}')
except AssertionError as e:
    print(f'✗ {e}')

### 题 2 🔧 简洁版对比：weight_decay = 0 vs 3

下面用 optimizer 的 weight_decay 参数实现（等价于从零版的 λ·l2_penalty）。先预测：

- weight_decay=0 和 3，哪个测试损失更低？
- 两个模型的 w 范数（大小）谁更大？

**【你的预测】**

In [ ]:
def train_concise(wd, num_epochs=100):
    net = nn.Sequential(nn.Linear(num_inputs, 1))
    for p in net.parameters():
        p.data.normal_()
    loss = nn.MSELoss()
    trainer = torch.optim.SGD([
        {"params": net[0].weight, "weight_decay": wd},
        {"params": net[0].bias}], lr=0.003)   # 偏置不衰减
    train_ls, test_ls = [], []
    for epoch in range(num_epochs):
        for X, y in train_iter:
            trainer.zero_grad()
            l = loss(net(X), y)
            l.backward()
            trainer.step()
        if (epoch + 1) % 20 == 0:
            train_ls.append(float(loss(net(train_features), train_labels)))
            test_ls.append(float(loss(net(test_features), test_labels)))
    print(f'weight_decay={wd}: 训练损失 {train_ls[-1]:.4f}, 测试损失 {test_ls[-1]:.4f}, w 范数 {float(net[0].weight.norm()):.2f}')
    return train_ls, test_ls, float(net[0].weight.norm())

In [ ]:
r0 = train_concise(0)   # 无正则
r3 = train_concise(3)   # 权重衰减

## 第二部分 · Dropout（暂退法）

### 题 3 🔧 从零实现 dropout_layer（TODO 4.2）

dropout 每次训练随机把一部分神经元输出置 0。补全实现，注意最后要除以 (1−dropout)。

In [ ]:
def dropout_layer(X, dropout):
    # TODO 4.2: mask = (torch.rand(X.shape) > dropout).float()；返回 mask * X / (1 - dropout)
    # 边界: dropout==0 返回 X；dropout==1 返回全 0
    raise NotImplementedError('⚠ TODO 4.2: dropout_layer 未完成')

In [ ]:
try:
    X = torch.arange(16, dtype=torch.float32).reshape(2, 8)
    print('原始:\n', X)
    print('dropout=0   (全保留):\n', dropout_layer(X, 0.0))
    print('dropout=0.5 (随机丢弃):\n', dropout_layer(X, 0.5))
    print('dropout=1   (全丢弃):\n', dropout_layer(X, 1.0))
    big = torch.rand(2000, 100)
    out = dropout_layer(big, 0.5)
    print('✓ 期望保持（均值应接近）:', round(big.mean().item(), 3), '≈', round(out.mean().item(), 3))
except NotImplementedError as e:
    print(f'⚠ {e}')

### 题 4 🔧 问答：train/eval 模式差异（面试高频经典 bug）

先预测再运行：

1. nn.Dropout 在训练模式和 eval 模式下行为有什么不同？
2. 为什么只在训练时 dropout？为什么 mask 要除以 (1−dropout)？
3. 忘了 net.eval() 会发生什么？

**【你的预测】**

In [ ]:
m = nn.Dropout(p=0.5)
x = torch.ones(1, 6)
m.train()
print('train 模式（丢弃 + 放大）:', m(x).tolist())
m.eval()
print('eval  模式（不丢弃）    :', m(x).tolist())

## 小结与面试衔接

- 权重衰减 = L2 正则，鼓励权重变小、防止过拟合；从零版加 λ·‖w‖²/2，简洁版用 weight_decay 参数
- Dropout 训练时随机丢弃神经元，推理时关闭（net.eval() 自动处理，忘了会导致推理结果随机变差）
- L1 vs L2（面试高频）：L1 让权重变精确的 0（稀疏），L2 让权重整体缩小（平滑）；几何上 L1 是菱形约束、L2 是圆
- 一轮「泛化与正则化」考点全覆盖：L1/L2、Dropout、BN（BN 在 ch07）